In [1]:
import pandas as pd
from transformers import T5Tokenizer , Trainer ,T5ForConditionalGeneration ,TrainingArguments

C:\Users\prith\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data =pd.read_csv("dataset/samsum-train.csv")
val_data = pd.read_csv("dataset/samsum-validation.csv")

In [3]:
train_data.sample(5)

,id,dialogue,summary
9172,13716810,Anita: Are you in Warsaw already?\r\nMelissa: ...,Anita will pick Melissa and Kelly up from the ...
2059,13821224,Abigail: <file_video>\r\nAbigail: Daily portio...,Abigail tries to cheer up Olivia and Kate with...
9114,13730946,"Leo: Hi J, how is it going?\r\nJames: not so b...",James has been there for 2 weeks and it's stil...
10835,13815889,Kylie: I have heard that you were with Alex La...,Eli was with Alex last night. His parents were...
4803,13728042,Adam: need a drink\r\nAdam: wanna go to Paddys...,Adam and Will are going to Paddys at 7.


In [4]:
train_data.shape

(14732, 3)

In [5]:
# ramdom sampling
train_data = train_data.sample(n = 4000 , random_state =42).reset_index(drop=True)
val_data = val_data.sample(n=500 ,random_state =42).reset_index(drop=True)

In [6]:
train_data.shape

(4000, 3)

# data preprocessing

In [7]:
import re # regex

def  clean_data(text):
    text = re.sub(r"\r\n"," ",text) # lines
    text = re.sub(r"\s+" ," ",text) # extra spaces
    text = re.sub(r"<.*?>"," ", text) #html
    text = text.strip().lower()
    return text

In [8]:
train_data["dialogue"] =train_data["dialogue"].apply(clean_data)
train_data["summary"] =train_data["summary"].apply(clean_data)

val_data["dialogue"] =val_data["dialogue"].apply(clean_data)
val_data["summary"] =val_data["summary"].apply(clean_data)

# Tokenize

In [9]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [12]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"] ,padding="max_length" ,max_length=512 ,truncation = True)
    targets = tokenizer(data["summary"] , padding="max_length" , max_length=150,truncation = True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [14]:
train_dataset = train_data.apply(tokenize , axis=1).tolist()
val_dataset = val_data.apply(tokenize , axis=1).tolist()

In [16]:
train_dataset[100]

{'input_ids': [66, 152, 10, 66, 182, 3, 12513, 179, 3, 184, 24, 68, 34, 31, 7, 341, 8, 7072, 13, 705, 145, 11140, 45, 284, 13008, 13, 8, 3, 1598, 5, 11, 48, 789, 1310, 435, 11140, 115, 14916, 12, 129, 8, 146, 102, 30, 1583, 5, 3, 17, 9, 1635, 9, 10, 11, 780, 34, 31, 7, 59, 21, 334, 712, 568, 16, 8, 3, 1598, 5, 59, 921, 523, 199, 45, 7813, 66, 152, 10, 3, 14504, 5, 3, 23, 317, 5, 3, 2, 3, 849, 162, 10, 43, 3, 9, 21842, 21, 8, 3, 29, 107, 7, 5, 12, 29, 23, 10, 3, 849, 162, 20, 6992, 207, 800, 3, 23, 152, 3, 10, 43, 3, 9, 3069, 3033, 52, 21, 3, 9, 3, 17, 52, 4215, 3709, 5, 3, 23, 152, 10, 3069, 8, 3, 29, 107, 7, 28, 8, 540, 62, 1097, 59, 11906, 3, 17, 52, 4215, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# Working with model

In [17]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1337.50it/s]


In [19]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [23]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_path = "./saved_model"

tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2031.42it/s]


## Test the core logic for summarization

In [24]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    
    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [32]:
test_dialogue = """ 

Dear Third-Year Engineering Students,

The Third-Year Engineering Internship is a compulsory academic component to be completed as per the applicable SPPU curriculum and departmental guidelines. Students are advised to take this internship seriously, not merely as an academic requirement, but as an opportunity to gain practical exposure, strengthen technical competencies, understand industry practices, and develop a foundation for their Final-Year Project, start-up opportunity and future placement preparation.

Please carefully follow the guidelines below:

1) Compulsory Internship – Plan Well in Advance:
   All Third-Year Engineering students are required to complete the prescribed internship of one month starting from January 2027 just after completion of end-semister university exams, followed by the evaluation prescribed by the applicable SPPU syllabus. Students must not wait until December/January to start searching for an internship. Early planning is essential for expected domain and to avoid last-minute difficulties.

2) Internship is an Academic Department Activity:
   The Third-Year Internship process, including identification of internship opportunities, company coordination, permissions, monitoring, documentation, attendance, evaluation, and academic requirements, will be handled by the respective dedicated Internship Faculty Coordinator.

3) Follow the Applicable SPPU Internship Guidelines:
   Students must carefully read and understand the internship guidelines, objectives, duration, documentation, attendance requirements, report format, evaluation methodology, and other requirements prescribed in the applicable SPPU syllabus/curriculum for their branch and pattern. Students must comply with the latest University and Institute guidelines applicable to their batch.

4) Start Searching for Internship Opportunities:
   Students should start approaching companies, industries, research organisations, MSMEs, startups and other relevant organisations well in advance. Do not wait for the department to provide an internship. Students should identify multiple suitable opportunities and initiate communication with prospective organisations early.

5) Select Internship Based on Your Technical Domain:
   Students should preferably select an internship aligned with their branch, technical interests, career goals, and areas in which they want to strengthen their knowledge and practical skills.
The objective should be to gain meaningful hands-on exposure rather than completing the internship only for obtaining a certificate.

6) Prefer Internship Opportunities that Provide Hands-on Learning:
   Students should give preference to internships where they can work on real industrial problems, engineering applications, software/tools, equipment, processes, design, testing, analysis, manufacturing, automation, research, or other relevant technical activities.
Students should clearly understand the work profile and expected learning outcomes before accepting an internship.

7) Explore Internship Opportunities Through Multiple Sources:
   Students can identify internship opportunities through:
   • Companies and industries in their technical domain
   • Senior students and alumni
   • Faculty contacts and professional networks
   • Previous internship records maintained by the department
   • Companies where previous students from department have successfully completed internships and their submitted internship reports.
   • Startups, MSMEs, research organisations and laboratories
   • Nearby industrial areas and MIDC locations in hometown
   • Professional networking and company websites
Students may also approach senior students to understand the nature of work, learning opportunities, company environment, and previous internship experience.

8) Utilise Nearby Industries and MIDC:
   Students who are travelling to or staying in their hometown during the internship period may explore suitable industries and organisations located in nearby MIDC/industrial areas.
Students should not restrict their search only to well-known companies or organisations in Pune. Relevant local industries can also provide valuable practical exposure.

9) Explore Internship Opportunities With Your Project Group:
   Students may, wherever feasible and academically appropriate, pursue internship opportunities together with their Final-Year Project group members.
Working together can help students understand industrial problem statements, identify potential project areas, divide responsibilities, and gain hands-on experience relevant to their Final-Year Major Project.

10) Try to Convert Internship into a Final-Year Project Opportunity:
    Students should actively explore opportunities where the internship organisation can provide:
    • Industry-sponsored projects
    • Real-world problem statements
    • Industry-defined challenges
    • Final-Year Major Project opportunities
    • Research and development assignments
    • Opportunities for continued project work after the internship
Students should begin working on a promising problem statement during the internship itself, wherever permitted by the organisation and department.

11) Focus on Skill Development:
    During the internship, students should consciously work on improving:
    • Technical knowledge
    • Practical and hands-on skills
    • Industry-relevant software/tools
    • Problem-solving ability
    • Engineering analysis and application
    • Communication and professional behaviour
    • Teamwork and time management
    • Documentation and technical reporting
Students should maintain a learning-oriented approach throughout the internship.

12) Verify the Internship Organisation Before Joining:
    Before accepting an internship, students should verify the organisation, nature of work, internship duration, work location/mode, reporting structure, technical activities, mentor details, and other relevant information.
Students should avoid accepting internships merely for the purpose of obtaining a certificate.

13) Online/Remote Internship Requires Prior Departmental Approval:
    Students intending to undertake an online/remote authentic internship as NPTEL, must obtain prior permission from the respective HoD.
The student must submit the required details, including organisation/company profile, mentor/supervisor details, internship duration, mode of internship, proposed work/project, learning outcomes, communication/contact details, and other information required by the department to establish the authenticity and academic relevance of the internship.
No online internship should be commenced without the required departmental approval.

14) Obtain Prior Permission for Departmental Documents:
    If a student requires a permission letter, recommendation letter, bonafide-related document, or any other official communication from the department for any private or government internship, the student should approach the respective Internship Faculty Coordinator well in advance with complete details of the organisation and internship.

15) Maintain Proper Internship Documentation:
    Students must maintain all documents related to the internship, including permission/approval letters, offer or confirmation letter, joining details, attendance, work records, mentor details, certificates, reports, and other documents prescribed by the department/SPPU.
Students should maintain a regular record of the work performed and learning achieved during the internship.

16) Internship Attendance and Professional Conduct:
    Students must maintain regular attendance and punctuality throughout the internship period and follow the rules, safety practices, confidentiality requirements, professional ethics, and discipline prescribed by the organisation.

17) Internship Evaluation – 100 Marks:
    The internship will be evaluated as per the applicable SPPU syllabus and departmental evaluation procedure, with the prescribed evaluation carrying 100 marks.
Students should therefore maintain proper records, demonstrate meaningful work and learning, complete the prescribed internship report/documentation, and appear for the evaluation as per the schedule communicated by the department.

18) Do Not Wait Until the Last Moment:
    Students are strongly advised to complete the identification and confirmation of their internship well before January 2027.
Last-minute searching may result in limited choices, unsuitable internship domains, difficulty in obtaining permission, or unnecessary inconvenience.

19) Take Guidance From the Department:
    For any clarification regarding internship selection, company suitability, permission, documentation, online internship, project opportunities, or evaluation, students should contact their respective Internship Faculty Coordinator/HoD.

20) Internship is an Opportunity – Make It Count:
    Students should consider this one-month internship as an important bridge between classroom learning and professional engineering practice. 
Students are advised to start and end one month internship in specific time frame suggested by SPPU, which starts immediately after end-semister university exams.

A meaningful internship can become the foundation for a strong Final-Year Project, better technical skills, initiative for start-up and improved preparedness for future placement opportunities.

All the best for your internship search and successful completion of the internship💐"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  the third-year engineering internship is a compulsory academic component to be completed as per the applicable sppu curriculum and departmental guidelines. students are advised to take this internship seriously, not as an academic requirement, but as an opportunity to gain practical exposure, strengthen technical competencies, understand industry practices, and develop a foundation for their final-year project, start-up opportunity and future placement preparation.
